# Phase 0 source audit companion

This executed notebook checks the retained four-file audit snapshot. It is not a production ingestion pipeline or a market findings report. Run from the project root or notebooks directory after restoring the files in outputs/audit/source_manifest.json. See docs/source_audit.md for interpretation and limits.

The complete effective-date mapping logic is in src/audit_mapping_and_time.py; the final cell reruns it. Source DST rows are not covered by the August sample. The notebook's captured outputs were produced by executing each code cell in a shared Python namespace; no notebook service is required.


In [ ]:
from pathlib import Path
import sys, json, hashlib
root = Path.cwd()
if not (root / 'PROGRESS.md').exists():
    root = root.parent
assert (root / 'PROGRESS.md').exists(), 'Run from project root or notebooks/'
sys.path.insert(0, str(root / 'src'))
from profile_audit_samples import load_samples, duplicate_check
from audit_mapping_and_time import trading_intervals
from datetime import date
frames = load_samples()
print({k: v.shape for k, v in frames.items()})

In [ ]:
keys = {'prices':['TradingDate','TradingPeriod','PointOfConnection'], 'reconciliation':['TradingDate','TradingPeriod','PointOfConnection','Network','Participant','FlowDirection'], 'generation':['Trading_Date','POC_Code','Nwk_Code','Gen_Code'], 'nsp':['NSP','Start date','Start TP']}
for source, columns in keys.items():
    assert not frames[source][columns].isna().any().any()
    result = duplicate_check(frames[source], columns)
    assert result['duplicate_rows_beyond_first'] == 0
    print(source, result)
print('Plant/date collisions:', duplicate_check(frames['generation'], ['Trading_Date','Gen_Code']))

In [ ]:
r, p = frames['reconciliation'], frames['prices']
key = ['PointOfConnection','TradingDate','TradingPeriod']
offtake = r.loc[r.FlowDirection == 'Offtake'].groupby(key, observed=True).KilowattHours.sum().reset_index()
joined = offtake.merge(p, on=key, how='left', validate='one_to_one')
assert len(joined) == len(offtake)
assert joined.DollarsPerMegawattHour.notna().all()
assert joined.KilowattHours.sum() == offtake.KilowattHours.sum()
print('Observed offtake kWh with unique matching price:', joined.KilowattHours.sum())
print('Unmatched price rows:', joined.DollarsPerMegawattHour.isna().sum())

In [ ]:
for day, expected in [('2026-04-05',50),('2026-09-27',46),('2026-08-01',48)]:
    intervals=trading_intervals(date.fromisoformat(day))
    assert len(intervals)==expected
    print(day, len(intervals), 'TP5:', intervals[4]['start_local'])
g=frames['generation']
assert g[['TP49','TP50']].isna().all().all()
assert g[[f'TP{i}' for i in range(1,49)]].notna().all().all()
print('August padding and populated period checks passed; actual DST market files remain untested.')

In [ ]:
# Release large frames before the complete effective-date audit reloads them.
del frames, r, p, g, offtake, joined
import gc
gc.collect()
from audit_mapping_and_time import main as run_mapping_audit
run_mapping_audit()